# Splitting the Wide class in two, NATIM

Wide is the largest class and the one labelled ubiquitous rather than assigned
to a layer. This notebook splits it in two on a single firing property and asks
whether the halves differ in their spatial neighbourhood.

**Why a median split rather than clustering.** A Gaussian mixture on the firing
metrics preferred four components for every class, not just Wide, with the BIC
gain scaling with sample size. That is what happens when the metrics are skewed
and heavy-tailed: the mixture adds components to approximate the shape rather
than finding clusters. There is no discovered cluster structure to report, so
this notebook makes a transparent descriptive division instead. Section 2 tests
whether any real structure exists, and the answer does not change what follows.

**Why this is not circular.** The split is defined by a firing property and then
tested against spatial arrangement. Those are independent measurements of the
same units, so a spatial difference is genuine evidence that the division
corresponds to something real.

Structure:
1. Load and restrict to quality-passing Wide units
2. Check whether cluster structure exists at all
3. Split on one axis, and check the halves do not differ in sorting quality
4. Characterise the two halves
5. Neighbour composition
6. Enrichment with both permutation nulls

In [ ]:
import os
os.chdir('/CSNG/studekat/ripple_paper_clean_copy/code_new_filter')

In [ ]:
from functions_analysis import *
import pandas as pd, numpy as np, yaml, pickle, itertools
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from statsmodels.stats.multitest import multipletests
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

In [ ]:
with open("/CSNG/studekat/ripple_paper_clean_copy/code_new_filter/params_analysis.yml") as f:
    params_analysis = yaml.safe_load(f)

DF_FOLDER = '/CSNG/studekat/ripple_paper_clean_copy/dataframes_new_filter'
MONKEY_LIST = ['N','F']
TYPE_REC = 'NATIM'
AREA = 'V12'
FINAL_CLASSES = params_analysis['final_classes']
CLASS_COLORS = params_analysis['colors_class']
CLASS_DICT = {'DOWN_narrow_shallow':'NarrBI','DOWN_narrow_sharp':'NarrTRI',
              'DOWN_wide':'Wide','DOWN_medium_shallow':'MedBI',
              'DOWN_medium_sharp':'MedTRI','UP':'Pos'}
KEY = ['monkey','date','array','cell_name']

TARGET = 'DOWN_wide'
QUALITY_LEVEL = 'pass_k3'
N_PERM = 1000
ALPHA = 0.05

# ---- THE SPLIT ----
# One interpretable axis, median split. LV_evoked is the default because
# firing regularity is the property that has separated classes most
# consistently elsewhere in this dataset. 'FR_baseline' and 'burst_index_evoked'
# are the obvious alternatives; rerun with each to check the spatial result is
# not specific to one choice.
SPLIT_ON = 'LV_evoked'
SPLIT_AT = 'median'          # 'median' or a number

## 1. Load

In [ ]:
def load_pkls(folder, monkeys, type_rec, tag=True):
    dfs = []
    for m in monkeys:
        for date in params_analysis['dates'][m][type_rec]:
            p = f'{DF_FOLDER}/{folder}/monkey{m}_all_arrays_date_{date}.pkl'
            try:
                with open(p,'rb') as f: d = pickle.load(f)
                if tag:
                    d['monkey'] = m; d['date'] = date
                dfs.append(d)
            except Exception:
                print(f'   missing {folder}: {m} {date}')
    return pd.concat(dfs, ignore_index=True) if dfs else None

df_wf = load_pkls('sua_prop_all_NATIM', MONKEY_LIST, TYPE_REC)
df_wf['area_merged'] = [a if a in ['V4','IT'] else 'V12' for a in df_wf['area']]
df_wf = df_wf[~df_wf['ch_is_noisy_100Hz'] & ~df_wf['ch_is_noisy_120Hz']]
df_wf = df_wf[df_wf['area_merged'] == AREA].reset_index(drop=True)

df_fire = load_pkls('sua_firing_NATIM', MONKEY_LIST, TYPE_REC, tag=False)

with open(f'{DF_FOLDER}/sua_quality_{TYPE_REC}/unit_inclusion_list.pkl','rb') as f:
    df_qual = pickle.load(f)

print('waveform:', df_wf.shape, '| firing:', df_fire.shape, '| quality:', df_qual.shape)

In [ ]:
fcols = [c for c in df_fire.columns if c not in df_wf.columns or c in KEY]
df = df_wf.merge(df_fire[fcols], on=KEY, how='left')

qcols = KEY + ['n_quality_pass'] + [c for c in df_qual.columns if c.startswith('pass_')]
qcols += [c for c in ['viol_ratio_min','coinc_ratio_worst','wf_shape_pc1_modesep']
          if c in df_qual.columns]
qcols = [c for c in dict.fromkeys(qcols) if c in df_qual.columns]
df = df.merge(df_qual[qcols], on=KEY, how='left')
df[QUALITY_LEVEL] = df[QUALITY_LEVEL].fillna(False).astype(bool)

print(f'{df.shape[0]} units in {AREA}, {int(df[QUALITY_LEVEL].sum())} passing {QUALITY_LEVEL}')
print()
print(df.loc[df[QUALITY_LEVEL],'final_class'].value_counts()
        .rename(index=CLASS_DICT).to_string())

## 2. Is there cluster structure to find?

Two checks. First, a Gaussian mixture on every class, not just Wide: if all
classes prefer the same number of components, the components track the shape of
the metric distribution rather than anything class-specific. Second, the same
fit on a null built by shuffling each metric independently, which preserves the
marginal distributions and destroys the dependence between them. If the null
also prefers more than one component, the components are an artifact of the
marginals.

Whatever this shows, the split below is a descriptive median division and makes
no claim of discovered clusters. The section exists so that claim is not made by
accident.

In [ ]:
SPLIT_METRICS = ['LV_evoked','CV2_evoked','burst_index_evoked',
                 'frac_spikes_in_burst_evoked','FR_baseline','FR_transient',
                 'modulation_index','psth_decay_ratio',
                 'first_spike_latency_s','first_spike_jitter_s']
SPLIT_METRICS = [m for m in SPLIT_METRICS if m in df.columns]

def prep_X(d, metrics):
    X = d[metrics].values.astype(float).copy()
    for i, m in enumerate(metrics):
        if m.startswith('FR_'):
            X[:, i] = np.log10(np.clip(X[:, i], 0, None) + 0.1)
    return StandardScaler().fit_transform(X)

d_ok = df[df[QUALITY_LEVEL]].copy()
d_w = d_ok[d_ok['final_class'] == TARGET].copy()
d_w = d_w.dropna(subset=SPLIT_METRICS + ['channel_order'])
print(f'{d_w.shape[0]} Wide units with complete firing metrics')

Xw = prep_X(d_w, SPLIT_METRICS)
rows = []
for cl in FINAL_CLASSES:
    d_c = d_ok[d_ok['final_class']==cl][SPLIT_METRICS].dropna()
    if d_c.shape[0] < 150:
        rows.append({'class': CLASS_DICT[cl], 'n': d_c.shape[0], 'k_best': None})
        continue
    Xc = prep_X(d_c, SPLIT_METRICS)
    b = [GaussianMixture(k, covariance_type='full', random_state=0,
                         n_init=3).fit(Xc).bic(Xc) for k in [1,2,3,4]]
    rows.append({'class': CLASS_DICT[cl], 'n': d_c.shape[0],
                 'k_best': [1,2,3,4][int(np.argmin(b))],
                 'gain_1v2': round(b[0]-b[1],1)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print('If every class prefers the same k and the gain scales with n, the')
print('components track the metric distribution rather than real clusters.')

In [ ]:
# marginal null: shuffle each metric independently, keeping the marginals
rng = np.random.default_rng(0)
Xnull = np.column_stack([rng.permutation(Xw[:, i]) for i in range(Xw.shape[1])])
for lab, X in [('Wide, observed', Xw), ('Wide, marginal null', Xnull)]:
    b = [GaussianMixture(k, covariance_type='full', random_state=0,
                         n_init=3).fit(X).bic(X) for k in [1,2,3,4]]
    print(f'{lab:22s} BIC {np.round(b,1)}  -> k={[1,2,3,4][int(np.argmin(b))]}')
print()
print('If the null prefers k > 1 as well, there is no cluster structure beyond')
print('what the marginal distributions alone produce.')

## 3. Split, and check the halves are not a quality artifact

Median split on the chosen axis. The critical check is that the two halves do
not differ in sorting quality: if they do, the split is a contamination axis and
any spatial difference would follow from data quality rather than biology.

In [ ]:
thr = d_w[SPLIT_ON].median() if SPLIT_AT == 'median' else float(SPLIT_AT)
d_w['wide_sub'] = np.where(d_w[SPLIT_ON] <= thr, 'Wide_lo', 'Wide_hi')
SUBS = ['Wide_lo', 'Wide_hi']
SUB_COLORS = {'Wide_lo': 'lightsteelblue', 'Wide_hi': 'navy'}

print(f'split on {SPLIT_ON} at {thr:.4f}')
print(d_w['wide_sub'].value_counts().to_string())
print()

qm = [c for c in ['viol_ratio_min','coinc_ratio_worst','wf_shape_pc1_modesep',
                  'n_quality_pass','n_spikes_total'] if c in d_w.columns]
if qm:
    print('sorting quality by half (these should NOT differ):')
    print(d_w.groupby('wide_sub')[qm].median().round(3).to_string())
    print()
    for c in qm:
        a = d_w.loc[d_w['wide_sub']==SUBS[0], c].dropna()
        b = d_w.loc[d_w['wide_sub']==SUBS[1], c].dropna()
        if len(a) > 10 and len(b) > 10:
            u = stats.mannwhitneyu(a, b)
            rb = 2*u.statistic/(len(a)*len(b)) - 1
            flag = '  <-- CONCERN' if abs(rb) > 0.2 else ''
            print(f'   {c:22s} rank-biserial {rb:+.3f}  p={u.pvalue:.2e}{flag}')
print()
ct = pd.crosstab(d_w['wide_sub'], d_w['monkey'])
print('by animal:'); print(ct.to_string())
chi2, p, _, _ = stats.chi2_contingency(ct)
print(f'animal association: chi2={chi2:.1f}, p={p:.2e}')

## 4. What else differs between the halves?

In [ ]:
rows = []
for m in SPLIT_METRICS:
    a = d_w.loc[d_w['wide_sub']==SUBS[0], m].dropna()
    b = d_w.loc[d_w['wide_sub']==SUBS[1], m].dropna()
    if len(a) < 10 or len(b) < 10: continue
    u = stats.mannwhitneyu(a, b)
    rows.append({'metric': m, f'median_{SUBS[0]}': round(np.median(a),4),
                 f'median_{SUBS[1]}': round(np.median(b),4),
                 'rank_biserial': round(2*u.statistic/(len(a)*len(b)) - 1, 3),
                 'p': u.pvalue})
df_sub = pd.DataFrame(rows)
df_sub['p_holm'] = multipletests(df_sub['p'], method='holm')[1]
df_sub = df_sub.reindex(df_sub['rank_biserial'].abs()
                        .sort_values(ascending=False).index)
print(df_sub.to_string(index=False))
print()
print(f'{SPLIT_ON} is at the top by construction. What matters is which other')
print('metrics follow it, since those indicate the axis is not isolated.')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), dpi=120)
for ax, m in zip(axes.flat, df_sub['metric'].tolist()[:6]):
    sns.violinplot(data=d_w, x='wide_sub', y=m, hue='wide_sub', order=SUBS,
                   palette=SUB_COLORS, inner='box', cut=0, ax=ax, legend=False)
    for v in ax.collections: v.set_alpha(0.7)
    if m.startswith('FR_'): ax.set_yscale('log')
    ax.set_xlabel('')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

# where the halves sit relative to the other classes
fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=120)
for ax, m in zip(axes, df_sub['metric'].tolist()[:3]):
    parts = []
    for cl in FINAL_CLASSES:
        if cl == TARGET: continue
        v = d_ok.loc[d_ok['final_class']==cl, m].dropna()
        if len(v) > 20: parts.append((CLASS_DICT[cl], v, CLASS_COLORS[cl]))
    for s in SUBS:
        parts.append((s.replace('Wide_',''), d_w.loc[d_w['wide_sub']==s, m].dropna(),
                      SUB_COLORS[s]))
    bp = ax.boxplot([p[1] for p in parts], labels=[p[0] for p in parts],
                    patch_artist=True, showfliers=False)
    for patch, p in zip(bp['boxes'], parts):
        patch.set_facecolor(p[2]); patch.set_alpha(0.75)
    ax.set_ylabel(m, fontsize=9); ax.tick_params(labelsize=7, rotation=45)
    if m.startswith('FR_'): ax.set_yscale('log')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 5. Neighbour composition

For a unit of each half, what fraction of its neighbours belong to each class.
Computed within array and date, then pooled. Neighbours are electrodes differing
by less than two positions in row and column, matching the main analysis, so
same-electrode neighbours are included.

In [ ]:
sub_map = dict(zip(map(tuple, d_w[KEY].values), d_w['wide_sub']))
CLASSES_EXT = [c for c in FINAL_CLASSES if c != TARGET] + SUBS

d_all = d_ok[d_ok['channel_order'] > -1].copy()
d_all['class_ext'] = [
    r_cls if r_cls != TARGET else sub_map.get(k)
    for r_cls, k in zip(d_all['final_class'],
                        map(tuple, d_all[KEY].values))]
n_drop = int(pd.isna(d_all['class_ext']).sum())
d_all = d_all[d_all['class_ext'].notna()]
print(f'{n_drop} Wide units dropped (no firing metrics, so not assigned a half)')
print(d_all['class_ext'].value_counts().to_string())

In [ ]:
def layout_coords(layout):
    c = np.full((64,2), -1.0)
    for ch in range(64):
        idx = np.where(layout == ch)
        if len(idx[0]): c[ch] = (idx[0][0], idx[1][0])
    return c

CI = {c: i for i, c in enumerate(CLASSES_EXT)}
NC = len(CLASSES_EXT)

def collect_blocks(d):
    """Per (date, array): class indices and neighbouring pair indices, as
    offsets into one global label vector so permutations act on all blocks."""
    labels, slices, IG, JG = [], [], [], []
    off = 0
    v_areas = {m: [a in ['V1','V2'] for a in params_analysis['areas'][m]]
               for m in d['monkey'].unique()}
    lay = {}
    for m in d['monkey'].unique():
        for par in ['odd','even']:
            lay[(m,par)] = layout_coords(
                np.array(params_analysis['layout'][f'{m}_{par}']))
    for (mk, date, array), g in d.groupby(['monkey','date','array'], sort=False):
        array = int(array)
        if not (1 <= array <= 16) or not v_areas[mk][array-1]:
            continue
        coords = lay[(mk, 'even' if array % 2 == 0 else 'odd')]
        ch = g['channel_order'].values.astype(int)
        ok = (ch >= 0) & (ch < 64)
        cls = g['class_ext'].map(CI).values[ok].astype(int)
        xy = coords[ch[ok]]
        good = xy[:,0] >= 0
        cls, xy = cls[good], xy[good]
        n = len(cls)
        if n < 2: continue
        i, j = np.triu_indices(n, k=1)
        nb = (np.abs(xy[i,0]-xy[j,0]) < 2) & (np.abs(xy[i,1]-xy[j,1]) < 2)
        IG.append(i[nb]+off); JG.append(j[nb]+off)
        labels.append(cls); slices.append((off, off+n)); off += n
    return (np.concatenate(labels), slices,
            np.concatenate(IG), np.concatenate(JG))

LAB, SLICES, IG, JG = collect_blocks(d_all)
print(f'{len(LAB)} units, {len(IG)} neighbouring pairs, {len(SLICES)} blocks')

In [ ]:
M_obs = np.zeros((NC, NC))
np.add.at(M_obs, (LAB[IG], LAB[JG]), 1)
np.add.at(M_obs, (LAB[JG], LAB[IG]), 1)
comp = M_obs / M_obs.sum(axis=1, keepdims=True)

names_ext = [CLASS_DICT.get(c, c).replace('Wide_','W_') for c in CLASSES_EXT]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), dpi=130)
sns.heatmap(M_obs, annot=True, fmt='.0f', cmap='Greys', ax=axes[0],
            xticklabels=names_ext, yticklabels=names_ext, annot_kws={'size':7})
axes[0].set_title('neighbouring pair counts')
sns.heatmap(comp, annot=True, fmt='.3f', cmap='viridis', ax=axes[1],
            xticklabels=names_ext, yticklabels=names_ext, annot_kws={'size':7})
axes[1].set_title('neighbour composition (rows sum to 1)')
for ax in axes:
    ax.set_ylabel('focal class'); ax.set_xlabel('neighbour class')
    ax.tick_params(labelsize=7)
plt.tight_layout(); plt.show()

i0, i1 = CI[SUBS[0]], CI[SUBS[1]]
cmp = pd.DataFrame({'neighbour': names_ext,
                    f'{SUBS[0]}_frac': comp[i0].round(4),
                    f'{SUBS[1]}_frac': comp[i1].round(4),
                    f'n_{SUBS[0]}': M_obs[i0].astype(int),
                    f'n_{SUBS[1]}': M_obs[i1].astype(int)})
cmp['ratio'] = (cmp[f'{SUBS[0]}_frac'] / cmp[f'{SUBS[1]}_frac']).round(3)
print(cmp.to_string(index=False))

tab = np.vstack([M_obs[i0], M_obs[i1]])
keep = tab.sum(axis=0) >= 10
chi2, p, dof, _ = stats.chi2_contingency(tab[:, keep])
V = np.sqrt(chi2 / tab[:, keep].sum())
print(f'\nneighbour profiles differ: chi2={chi2:.1f}, dof={dof}, p={p:.2e}')
print(f"Cramer's V = {V:.4f}   (the effect size; p will be small at this n)")

## 6. Enrichment with both nulls

`global` shuffles class labels across every block; `within` shuffles only inside
one array on one date. A pair significant under `within` reflects local
adjacency rather than shared array territory.

In [ ]:
PAIRS_EXT = ["+".join(sorted(t)) for t in itertools.combinations(CLASSES_EXT, 2)]
PAIRS_EXT += ["+".join((c,c)) for c in CLASSES_EXT]
PIE = {p: i for i, p in enumerate(PAIRS_EXT)}
LUT = np.zeros((NC, NC), dtype=int)
for i, a in enumerate(CLASSES_EXT):
    for j, b in enumerate(CLASSES_EXT):
        LUT[i,j] = PIE["+".join(sorted((a,b)))]

obs_close = np.bincount(LUT[LAB[IG], LAB[JG]], minlength=len(PAIRS_EXT))
obs_all = np.zeros(len(PAIRS_EXT), dtype=np.int64)
for a, b in SLICES:
    seg = LAB[a:b]; n = len(seg)
    if n < 2: continue
    i, j = np.triu_indices(n, k=1)
    obs_all += np.bincount(LUT[seg[i], seg[j]], minlength=len(PAIRS_EXT))

def run_perm(lab, slices, ig, jg, scope, n_perm, seed=0):
    rng = np.random.default_rng(seed)
    out = np.zeros((n_perm, len(PAIRS_EXT)), dtype=np.int64)
    l = lab.copy()
    for p in range(n_perm):
        if scope == 'global':
            rng.shuffle(l)
        else:
            for a, b in slices:
                if b-a > 1:
                    s = l[a:b]; rng.shuffle(s); l[a:b] = s
        out[p] = np.bincount(LUT[l[ig], l[jg]], minlength=len(PAIRS_EXT))
    return out

RESULTS = {}
for scope in ['global','within']:
    print(f'{scope} null, {N_PERM} permutations ...')
    perm = run_perm(LAB, SLICES, IG, JG, scope, N_PERM)
    assert np.all(perm.sum(axis=1) == obs_close.sum()), 'null not label-preserving'
    rows = []
    for p, i in PIE.items():
        o = obs_close[i]; nl = perm[:, i]
        e, sd = float(nl.mean()), float(nl.std())
        n_ge = int((nl >= o).sum()); n_le = int((nl <= o).sum())
        rows.append({'pair': p, 'n_all': int(obs_all[i]), 'n_close': int(o),
                     'exp': round(e,2),
                     'enrichment': round(o/e,3) if e > 0 else np.nan,
                     'z': round((o-e)/sd,3) if sd > 0 else np.nan,
                     'p_perm': min(2*min(n_ge+1, n_le+1)/(N_PERM+1), 1.0)})
    t = pd.DataFrame(rows)
    t['p_holm'] = multipletests(t['p_perm'], method='holm')[1]
    t['pair_short'] = t['pair'].apply(
        lambda s: "+".join(CLASS_DICT.get(x, x).replace('Wide_','W_')
                           for x in s.split('+')))
    RESULTS[scope] = t.sort_values('z', ascending=False)
    print(f'   {int((t["p_holm"] < ALPHA).sum())}/{len(t)} significant')

In [ ]:
for scope in ['global','within']:
    t = RESULTS[scope]
    m = t['pair'].apply(lambda s: any(x in SUBS for x in s.split('+')))
    print(f'===== {scope} null, pairs involving a Wide half =====')
    print(t[m][['pair_short','n_all','n_close','exp','enrichment','z','p_holm']]
          .to_string(index=False))
    print()

In [ ]:
g = RESULTS['global'].set_index('pair')
w = RESULTS['within'].set_index('pair')
rows = []
for other in CLASSES_EXT:
    r = {'partner': CLASS_DICT.get(other, other).replace('Wide_','W_')}
    for s in SUBS:
        k = "+".join(sorted((s, other)))
        r[f'{s}_global'] = g.loc[k,'enrichment'] if k in g.index else np.nan
        r[f'{s}_within'] = w.loc[k,'enrichment'] if k in w.index else np.nan
        r[f'{s}_n'] = int(g.loc[k,'n_all']) if k in g.index else 0
    rows.append(r)
df_prof = pd.DataFrame(rows)
print(df_prof.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4), dpi=120)
x = np.arange(len(df_prof))
for ax, scope in zip(axes, ['global','within']):
    for k, s in enumerate(SUBS):
        ax.bar(x + (k-0.5)*0.35, df_prof[f'{s}_{scope}'], width=0.35,
               color=SUB_COLORS[s], alpha=0.85, label=s)
    ax.axhline(1, color='k', ls='--', lw=1)
    ax.set_xticks(x); ax.set_xticklabels(df_prof['partner'], rotation=45, fontsize=8)
    ax.set_ylabel('enrichment'); ax.set_title(f'{scope} null')
    ax.legend(frameon=False, fontsize=8)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 7. Robustness: does the result depend on which axis was used?

The split is a choice. Rerunning on the other candidate axes shows whether the
spatial difference is a property of the units or of this particular division.

In [ ]:
ALT_AXES = [a for a in ['LV_evoked','FR_baseline','burst_index_evoked',
                        'CV2_evoked'] if a in d_w.columns]

def split_and_profile(axis):
    """Redo the whole split on a different axis and return the effect size."""
    thr_a = d_w[axis].median()
    smap = dict(zip(map(tuple, d_w[KEY].values),
                    np.where(d_w[axis] <= thr_a, 'Wide_lo', 'Wide_hi')))
    d2 = d_ok[d_ok['channel_order'] > -1].copy()
    d2['class_ext'] = [
        cl if cl != TARGET else smap.get(k)
        for cl, k in zip(d2['final_class'], map(tuple, d2[KEY].values))]
    d2 = d2[d2['class_ext'].notna()]
    L2, S2, I2, J2 = collect_blocks(d2)
    M2 = np.zeros((NC, NC))
    np.add.at(M2, (L2[I2], L2[J2]), 1)
    np.add.at(M2, (L2[J2], L2[I2]), 1)
    t2 = np.vstack([M2[CI['Wide_lo']], M2[CI['Wide_hi']]])
    k2 = t2.sum(axis=0) >= 10
    if k2.sum() < 2:
        return None
    c2, p2, _, _ = stats.chi2_contingency(t2[:, k2])
    return {'split_axis': axis, 'threshold': round(float(thr_a), 4),
            'chi2': round(c2, 1),
            'cramers_V': round(float(np.sqrt(c2 / t2[:, k2].sum())), 4),
            'p': f'{p2:.2e}',
            'n_pairs_lo': int(M2[CI['Wide_lo']].sum()),
            'n_pairs_hi': int(M2[CI['Wide_hi']].sum())}

alt_rows = [r for r in (split_and_profile(a) for a in ALT_AXES) if r]
df_alt = pd.DataFrame(alt_rows)
print(df_alt.to_string(index=False))
print()
print("A similar Cramer's V across axes means the two halves of Wide differ")
print('spatially however the division is drawn, which is the stronger result.')
print('An effect for only one axis means the finding belongs to that axis.')

## 8. Export

In [ ]:
out_dir = f'{DF_FOLDER}/wide_split_{TYPE_REC}'
ensure_dir_exists(out_dir)
d_w[KEY + ['wide_sub', SPLIT_ON]].to_pickle(f'{out_dir}/wide_half_labels.pkl')
df_sub.to_csv(f'{out_dir}/half_firing_differences.csv', index=False)
cmp.to_csv(f'{out_dir}/neighbour_composition.csv', index=False)
for scope in RESULTS:
    RESULTS[scope].to_csv(f'{out_dir}/enrichment_{scope}.csv', index=False)
df_prof.to_csv(f'{out_dir}/half_enrichment_profiles.csv', index=False)
if len(df_alt):
    df_alt.to_csv(f'{out_dir}/split_axis_robustness.csv', index=False)
print('saved to', out_dir)
print(f'split: {SPLIT_ON} at {thr:.4f}')

## How to read this

**Section 2 sets the framing, not the method.** A mixture model on these metrics
prefers several components for every class, with the gain scaling with sample
size, so there is no cluster structure to report. The split below is descriptive
and should be described that way in any figure caption: Wide divided at the
median of one firing property, not two discovered subtypes.

**Section 3 is the gate.** If the halves differ in `viol_ratio_min`,
`coinc_ratio_worst` or `wf_shape_pc1_modesep`, the split is tracking sorting
quality and any spatial difference follows from that rather than from biology.
Rank-biserial values above about 0.2 are flagged.

**Section 5 answers the question simply.** Neighbour composition per half, with
Cramer's V as the effect size, since p will be small at these counts regardless.

**Section 6 is the rigorous version.** Read the `within` null for local
adjacency. Watch `n_all`: splitting Wide halves the pair counts for every
combination involving it, so previously well-sampled pairs become noisier.

**Section 7 is what makes the result defensible.** If the halves differ
spatially however the division is drawn, that is a property of the units. If
only one axis gives an effect, the finding belongs to that axis.

**On interpretation.** Wide is labelled ubiquitous because it spans layers, so a
plausible reading of any spatial difference is laminar rather than a cell-type
division, with the neighbour profiles reflecting which classes share those
layers.